# MXINT8 Convolution With A Triton Accumulator

This notebook shows a different path from Microsoft `microxcaling`'s normal emulation flow.

Instead of quantizing to MX-representable float tensors and then calling PyTorch convolution, this example keeps raw MX-style integer elements plus shared scales, runs convolution in a Triton kernel, applies the scales inside the accumulator, and returns the final float output.

The implementation lives in `mx_triton_conv2d.py`.

## Important Scope

This example uses MXINT8-style blocks:

```text
real_value ~= int8_element * shared_block_scale
```

That makes the accumulator easy to inspect because the Triton kernel multiplies integer elements by activation and weight scales during convolution. This is not the same as Microsoft's `mx.Conv2d`, which quantizes tensors to MX-compatible float values and then uses PyTorch convolution internally.

Current example limitations: CUDA only, `Conv2d` only, NCHW input, OIHW weights, `groups == 1`, `dilation == 1`, and zero padding only.

## Install Requirements

Run in a CUDA environment:

```bash
pip install torch triton
```

Microsoft's repo is still useful for MX format context and comparison:

```bash
git clone https://github.com/microsoft/microxcaling.git
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from mx_triton_conv2d import (
    MXInt8TritonConv2d,
    mxint8_conv2d_triton,
    quantize_mxint8_channel_blocks,
    replace_conv2d_with_mxint8_triton,
)

## Low-Level Kernel Usage

This path makes the data movement explicit: quantize activations and weights to raw MXINT8-style tensors, then pass those raw elements and scales to the Triton convolution.

In [ ]:
device = "cuda"
torch.manual_seed(0)

x = torch.randn(1, 32, 16, 16, device=device)
weight = torch.randn(8, 32, 3, 3, device=device)
bias = torch.randn(8, device=device)

x_mx = quantize_mxint8_channel_blocks(x, axis=1, block_size=32)
w_mx = quantize_mxint8_channel_blocks(weight, axis=1, block_size=32)

y_mx = mxint8_conv2d_triton(x_mx, w_mx, bias, stride=1, padding=1)
y_ref = F.conv2d(x, weight, bias, stride=1, padding=1)

print("raw activation elements:", x_mx.elements.dtype, tuple(x_mx.elements.shape))
print("activation scales:", x_mx.scales.dtype, tuple(x_mx.scales.shape))
print("raw weight elements:", w_mx.elements.dtype, tuple(w_mx.elements.shape))
print("weight scales:", w_mx.scales.dtype, tuple(w_mx.scales.shape))
print("output shape:", tuple(y_mx.shape))
print("mean abs diff vs fp32:", (y_ref - y_mx).abs().mean().item())

## Model Replacement Usage

For a model-level test, replace supported `nn.Conv2d` layers with `MXInt8TritonConv2d`. Each forward pass quantizes the input activation and current weight tensor, then runs the Triton accumulator convolution.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 8 * 8, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = TinyCNN().eval().to(device)
mx_model = replace_conv2d_with_mxint8_triton(model, block_size=32).eval().to(device)

for name, module in mx_model.named_modules():
    if isinstance(module, MXInt8TritonConv2d):
        print(name, "->", module.__class__.__name__)

In [ ]:
sample = torch.randn(4, 3, 16, 16, device=device)

with torch.no_grad():
    y_baseline = model(sample)
    y_accumulator = mx_model(sample)

print("baseline output shape:", tuple(y_baseline.shape))
print("Triton MX accumulator output shape:", tuple(y_accumulator.shape))
print("mean abs diff:", (y_baseline - y_accumulator).abs().mean().item())

## What To Inspect Next

Open `mx_triton_conv2d.py` and look at `_mxint8_conv2d_kernel`. The accumulator line is the core behavior:

```python
acc += x_elem * w_elem * x_scale * w_scale
```

That means full tensors are not dequantized before convolution. The kernel loads raw int8 elements and shared scales, combines them per multiply-accumulate, and writes the final float32 convolution output.